In [ ]:
# === CHALLENGE 3 (GLP1R) : TARGET SETUP ===## Builds the cropped active-state receptor from PDB 6X18, and saves the geometry needed# to ask the only question that matters here: how DEEP does a candidate reach.## GLP1R is a class B1 GPCR and works by a two-domain mechanism. The hormone's C-terminal# half grips the extracellular domain; its first eight residues thread ~40 A down into# the seven-helix bundle, and that is what activates the receptor. Removing those eight# residues gives exendin 9-39, a real drug that binds well and ANTAGONISES. So a design# that only binds the extracellular groove - which is what the challenge page recommends,# and what a folded minibinder would naturally do - cannot win, because the winner has to# activate.## Measured from 6X18 itself: GLP-1 spans 39.6 A of depth, makes 22 extracellular contacts# and 30 transmembrane ones, and 30 residues of alpha helix rises about 45 A. The agonist# is essentially one long helix threading a channel, not a globular domain sitting on a# surface.import glob, json, os, shutil, subprocess, sys, timet0 = time.time()def sh(cmd, check=False):    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)    if r.returncode != 0:        print("FAILED:", cmd)        print((r.stdout + r.stderr)[-1000:])        if check:            raise SystemExit("setup failed")    return rprint(sys.version.split()[0], flush=True)sh("nvidia-smi --query-gpu=name --format=csv,noheader")os.makedirs("/kaggle/working/params", exist_ok=True)src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)if src:    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)    for f in src:        dst = "/kaggle/working/params/" + os.path.basename(f)        if not os.path.exists(dst):            shutil.copy(f, dst)else:    print("downloading AF2 params (~3.5 GB) ...", flush=True)    sh("wget -q -O /kaggle/working/af2.tar "       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar", check=True)    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)sh("pip install -q biopython")import urllib.requestimport numpy as npfrom Bio.PDB import PDBParser, PDBIO, Selectfrom Bio.PDB.Polypeptide import protein_letters_3to1 as T31os.makedirs("/kaggle/working/targets", exist_ok=True)raw = "/kaggle/working/6X18.pdb"if not os.path.exists(raw):    open(raw, "wb").write(urllib.request.urlopen(        "https://files.rcsb.org/download/6X18.pdb", timeout=180).read())RADIUS, MAXGAP, ECD_END = 10.0, 4, 145mdl = PDBParser(QUIET=True).get_structure("x", raw)[0]rec = [r for r in mdl["R"] if r.id[0] == " "]pep = [r for r in mdl["P"] if r.id[0] == " "]native_seq = "".join(T31.get(r.get_resname(), "X") for r in pep)print("6X18 chain R: %d residues; chain P (GLP-1): %d-%d %s"      % (len(rec), pep[0].id[1], pep[-1].id[1], native_seq), flush=True)if native_seq[0] != "H" or pep[0].id[1] != 7:    raise SystemExit("chain P does not start at His7 - the depth reference would be wrong")patoms = np.array([a.get_coord() for r in pep for a in r])mind, ratoms = {}, {}for r in rec:    ra = np.array([a.get_coord() for a in r])    ratoms[r.id[1]] = ra    mind[r.id[1]] = float(np.min(np.linalg.norm(ra[:, None, :] - patoms[None, :, :], axis=-1)))keep = sorted(k for k, v in mind.items() if v <= RADIUS)present = {r.id[1] for r in rec}filled = set(keep)for a, b in zip(keep, keep[1:]):    if 1 < b - a <= MAXGAP + 1 and all(x in present for x in range(a + 1, b)):        filled.update(range(a + 1, b))kept = sorted(filled)contacts = set()for r in pep:    ra = np.array([a.get_coord() for a in r])    for rid, rra in ratoms.items():        if np.min(np.linalg.norm(ra[:, None, :] - rra[None, :, :], axis=-1)) <= 4.5:            contacts.add(rid)lost = sorted(contacts - filled)if lost:    raise SystemExit("crop drops GLP-1 contacts %s - widen RADIUS" % lost)print("cropped target: %d residues, all %d GLP-1 contacts retained"      % (len(kept), len(contacts)), flush=True)class Keep(Select):    def accept_chain(self, c): return c.id == "R"    def accept_residue(self, r): return r.id[0] == " " and r.id[1] in filled    def accept_atom(self, a): return a.element != "H"TARGET = "/kaggle/working/targets/GLP1R_crop.pdb"io = PDBIO(); io.set_structure(mdl); io.save(TARGET, Keep())# Depth axis: extracellular-domain centroid -> transmembrane centroid, so one number per# atom says how far into the bundle it sits. Saved so every later run uses the same axis.def ca(r): return r["CA"].get_coord() if "CA" in r else Nonekept_res = [r for r in rec if r.id[1] in filled]ecd_c = np.mean([ca(r) for r in kept_res if r.id[1] <= ECD_END and ca(r) is not None], axis=0)tm_c = np.mean([ca(r) for r in kept_res if r.id[1] > ECD_END and ca(r) is not None], axis=0)axis = (tm_c - ecd_c) / np.linalg.norm(tm_c - ecd_c)native_depth = {}for r in pep:    c = ca(r)    if c is not None:        native_depth[r.id[1]] = round(float(np.dot(c - ecd_c, axis)), 2)ref_deep = max(native_depth.values())print("GLP-1 native depth: H7 at %.1f A, C-terminus at %.1f A, span %.1f A"      % (native_depth[7], native_depth[pep[-1].id[1]], ref_deep - min(native_depth.values())),      flush=True)cmap = [dict(uniprot=r.id[1], aa=T31.get(r.get_resname(), "X"),             depth=round(float(np.dot(ca(r) - ecd_c, axis)), 2) if ca(r) is not None else None,             region="ecd" if r.id[1] <= ECD_END else "tm",             contact=r.id[1] in contacts)        for r in kept_res]json.dump(dict(order=cmap, ecd_end=ECD_END, radius=RADIUS,               ecd_centroid=[float(x) for x in ecd_c], axis=[float(x) for x in axis],               native_glp1=dict(seq=native_seq, start=pep[0].id[1], depth=native_depth),               tm_contacts=sorted(x for x in contacts if x > ECD_END),               ecd_contacts=sorted(x for x in contacts if x <= ECD_END)),          open("/kaggle/working/glp1r_map.json", "w"), indent=1)TM_HOT = ",".join("R%d" % x for x in sorted(x for x in contacts if x > ECD_END))ECD_HOT = ",".join("R%d" % x for x in sorted(x for x in contacts if x <= ECD_END))open("/kaggle/working/glp1r_tm_hotspot.txt", "w").write(TM_HOT)open("/kaggle/working/glp1r_ecd_hotspot.txt", "w").write(ECD_HOT)print("activation hotspots (transmembrane): %d" % len(TM_HOT.split(",")), flush=True)print("binding hotspots (extracellular)   : %d" % len(ECD_HOT.split(",")), flush=True)# A second, much smaller target for the BINDING job. A design that only grips the# extracellular landing pad never sees the helical bundle, so making it carry 192# residues is paying for geometry it cannot touch. Measured on 6X18: an extracellular-only# shell at 12 A keeps 76 residues in 5 segments and retains all 22 contacts the hormone# makes there - smaller than the 122-residue TNF target, so these runs are cheap.ECD_RADIUS = 12.0ekeep = sorted(k for k, v in mind.items() if v <= ECD_RADIUS and k <= ECD_END)efilled = set(ekeep)for a, b in zip(ekeep, ekeep[1:]):    if 1 < b - a <= MAXGAP + 1 and all(x in present for x in range(a + 1, b)):        efilled.update(range(a + 1, b))ecd_contacts = sorted(x for x in contacts if x <= ECD_END)elost = [x for x in ecd_contacts if x not in efilled]if elost:    raise SystemExit("extracellular crop drops contacts %s - widen ECD_RADIUS" % elost)class KeepECD(Select):    def accept_chain(self, c): return c.id == "R"    def accept_residue(self, r): return r.id[0] == " " and r.id[1] in efilled    def accept_atom(self, a): return a.element != "H"ECD_TARGET = "/kaggle/working/targets/GLP1R_ecd_crop.pdb"io2 = PDBIO(); io2.set_structure(mdl); io2.save(ECD_TARGET, KeepECD())ecd_order = [dict(uniprot=r.id[1], aa=T31.get(r.get_resname(), "X"),                  contact=r.id[1] in contacts)             for r in rec if r.id[1] in efilled]json.dump(dict(order=ecd_order, radius=ECD_RADIUS, ecd_end=ECD_END,               contacts=ecd_contacts,               seq="".join(d["aa"] for d in ecd_order)),          open("/kaggle/working/glp1r_ecd_map.json", "w"), indent=1)print("extracellular-only target: %d residues, all %d of its contacts retained"      % (len(efilled), len(ecd_contacts)), flush=True)PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "         "print(jax.__version__)\"")r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):    print("installing jax 0.4.38 ...", flush=True)    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")chk = subprocess.run(    "cd /kaggle/working && python -c \""    "import jax; from colabdesign import mk_afdesign_model; "    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "    "m.prep_inputs(pdb_filename='%s', chain='R', binder_len=30, hotspot='%s'); "    "print('READY', jax.__version__, jax.devices()); "    "print('target length seen by the model:', m._target_len)\"" % (TARGET, TM_HOT),    shell=True, capture_output=True, text=True)print(chk.stdout[-600:])if chk.returncode != 0:    print("PREP FAILED:"); print(chk.stderr[-1500:])print("GLP1R SETUP DONE in %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# === CHALLENGE 3 (GLP1R) : GENERATION, TWO JOBS AT ONCE ===## Two different design problems run in parallel, one per T4, because they need different# targets and there is no point rebuilding a model between them.##   GPU 0, "ecd"    - bind the extracellular landing pad. 76-residue target, so cheap.#                     This is the job our pipeline already does well, and the one the lab#                     can measure reliably. It yields an ANTAGONIST by mechanism.#   GPU 1, "thread" - thread the helical bundle, which is what actually activates the#                     receptor. 192-residue target, long thin binder. A declared long shot.## WHAT WE CAN AND CANNOT SCREEN FOR. Yesterday's control run (cellGLP2_control.py) tested# whether AlphaFold can tell the real hormone from the truncated drug that blocks the# receptor. It cannot: 34.1 A against 33.3 A, inside our noise, and a scrambled nonsense# sequence reached 37.0 A - deeper than either. So depth MAGNITUDE is not a selection rule# and nothing here pretends otherwise.## What the same run did separate is depth POLARITY - which end of the peptide points down# the channel. Both real peptides put their N-terminus at the deep end to within 0.1 A;# the scrambled control had its N-terminus at -7.4 A while its deepest atom sat at 37.0 A,# i.e. sprawled in backwards. That is a real "is this threaded at all" filter, and it is# the only thing we screen threaders on besides confidence. Activation itself is UNSCREENED# and the methods file has to say so.## Also measured for every design, because the competition records it and we only found out# late: mean Kyte-Doolittle hydropathy. Our TNF designs were greasy enough to have failed# that screen. Measuring it during generation rather than after means we can select on it.import os, subprocess, sys, textwrap, timeTIME_BUDGET_MIN = 420          # 7 h wall clock; Kaggle sessions cap at 12SOFT, TEMP, HARD = 90, 40, 20WORKER = r'''import json, os, random, sys, timeGPU, MODE, BUDGET = sys.argv[1], sys.argv[2], float(sys.argv[3])os.environ["CUDA_VISIBLE_DEVICES"] = GPU          # must precede the jax importSOFT, TEMP, HARD = int(sys.argv[4]), int(sys.argv[5]), int(sys.argv[6])t0 = time.time()def log(m): print("[%s] %s [%.1f min]" % (MODE, m, (time.time()-t0)/60), flush=True)import numpy as npfrom colabdesign import mk_afdesign_model, clear_memFULL = json.load(open("/kaggle/working/glp1r_map.json"))AXIS = np.array(FULL["axis"]); ECD_C = np.array(FULL["ecd_centroid"])AXIS_OFF = float(ECD_C @ AXIS)if MODE == "ecd":    M = json.load(open("/kaggle/working/glp1r_ecd_map.json"))    TARGET = "/kaggle/working/targets/GLP1R_ecd_crop.pdb"    HOTSPOT = open("/kaggle/working/glp1r_ecd_hotspot.txt").read().strip()    LEN_MIN, LEN_MAX = 55, 80    WANT = set(M["contacts"])else:    M = FULL    TARGET = "/kaggle/working/targets/GLP1R_crop.pdb"    HOTSPOT = open("/kaggle/working/glp1r_tm_hotspot.txt").read().strip()    LEN_MIN, LEN_MAX = 30, 45    WANT = set(FULL["tm_contacts"])ORDER = M["order"]TSEQ = "".join(d["aa"] for d in ORDER)log("target %s, %d residues, %d hotspots, binder %d-%d"    % (os.path.basename(TARGET), len(ORDER), len(HOTSPOT.split(",")), LEN_MIN, LEN_MAX))# Kyte-Doolittle hydropathy. The competition records the mean of this per design, and a# greasy sequence reads as an aggregation risk, so it is a selection variable not a note.KD = dict(A=1.8, R=-4.5, N=-3.5, D=-3.5, C=2.5, Q=-3.5, E=-3.5, G=-0.4, H=-3.2, I=4.5,          L=3.8, K=-3.9, M=1.9, F=2.8, P=-1.6, S=-0.8, T=-0.7, W=-0.9, Y=-1.3, V=4.2)def hydropathy(seq):    vals = [KD[c] for c in seq if c in KD]    return sum(vals) / len(vals) if vals else 0.0def dedupe(chain):    """AfDesign output carries each residue twice; keep the first copy of each number."""    res, seen = [], set()    for r in list(chain):        if r.id[0] != " " or r.id[1] in seen:            chain.detach_child(r.id); continue        seen.add(r.id[1]); res.append(r)    return resdef measure(pdb_path, blen):    from Bio.PDB import PDBParser    from Bio.PDB.Polypeptide import protein_letters_3to1 as T31    mdl = PDBParser(QUIET=True).get_structure("c", pdb_path)[0]    kept = {ch.id: dedupe(ch) for ch in mdl}    # Identify the target by its LENGTH, not by "whichever chain is smaller". The    # extracellular target is 76 residues and a binder may be 80, so smaller-is-binder    # would silently swap them and every number below would describe the wrong molecule.    tid = None    for cid, rs in kept.items():        if len(rs) == len(ORDER):            s = "".join(T31.get(r.get_resname(), "X") for r in rs)            if s == TSEQ:                tid = cid; break    if tid is None:        raise ValueError("no chain matches the target sequence; chains %s"                         % {k: len(v) for k, v in kept.items()})    bid = [c for c in kept if c != tid]    if len(bid) != 1:        raise ValueError("expected exactly one binder chain, got %s" % bid)    bres = kept[bid[0]]    if len(bres) != blen:        raise ValueError("binder chain %d residues, expected %d" % (len(bres), blen))    batoms = np.array([a.get_coord() for r in bres for a in r])    hits = []    for k, r in enumerate(kept[tid]):        ra = np.array([a.get_coord() for a in r])        if np.min(np.linalg.norm(ra[:, None, :] - batoms[None, :, :], axis=-1)) <= 5.0:            hits.append(ORDER[k]["uniprot"])    rec = dict(epitope=sorted(hits), on_site=sorted(set(hits) & WANT))    # Helicity without DSSP: in an alpha helix CA(i) to CA(i+4) is about 6.2 A.    cas = [r["CA"].get_coord() for r in bres if "CA" in r]    if len(cas) > 4:        d = [float(np.linalg.norm(cas[i] - cas[i + 4])) for i in range(len(cas) - 4)]        rec["helicity"] = round(sum(1 for x in d if 5.5 <= x <= 7.0) / len(d), 3)    if MODE == "thread":        proj = batoms @ AXIS - AXIS_OFF        nterm = np.array([a.get_coord() for a in bres[0]]) @ AXIS - AXIS_OFF        cterm = np.array([a.get_coord() for a in bres[-1]]) @ AXIS - AXIS_OFF        rec["deepest"] = round(float(np.max(proj)), 1)        rec["nterm_depth"] = round(float(np.max(nterm)), 1)        rec["cterm_depth"] = round(float(np.max(cterm)), 1)        # the polarity screen: is the N-terminal end actually the deep end        rec["threaded"] = bool(rec["nterm_depth"] >= rec["deepest"] - 5.0                               and rec["nterm_depth"] > rec["cterm_depth"])        tm = ecd = 0        for k, r in enumerate(kept[tid]):            ra = np.array([a.get_coord() for a in r])            if np.min(np.linalg.norm(ra[:, None, :] - batoms[None, :, :], axis=-1)) <= 4.5:                if ORDER[k].get("region") == "tm":                    tm += 1                else:                    ecd += 1        rec["tm_contacts"], rec["ecd_contacts"] = tm, ecd    return recoutdir = "/kaggle/working/glp1r_%s" % MODEos.makedirs(outdir, exist_ok=True)rec_path = "%s/designs.jsonl" % outdirrng = random.Random(9000 if MODE == "ecd" else 9500)made = 0i = 0while (time.time() - t0) / 60 < BUDGET:    i += 1    name = "glp1r_%s_%03d" % (MODE, i)    L = rng.randint(LEN_MIN, LEN_MAX)    try:        clear_mem()        model = mk_afdesign_model(protocol="binder", data_dir="/kaggle/working")        model.prep_inputs(pdb_filename=TARGET, chain="R", binder_len=L, hotspot=HOTSPOT)        model.design_3stage(soft_iters=SOFT, temp_iters=TEMP, hard_iters=HARD)        seq = model.get_seqs()[0]        design_iptm = float(model.aux["log"].get("i_ptm", 0.0))        # The score that counts is an INDEPENDENT re-prediction of the discrete sequence,        # never the optimiser's own number, which is optimistic by construction.        try:            model.predict(seq=seq, num_recycles=3, verbose=False)        except TypeError:            model.predict(seq=seq, verbose=False)        lg = model.aux["log"]        r = dict(name=name, mode=MODE, length=L, sequence=seq,                 design_iptm=round(design_iptm, 3),                 true_iptm=round(float(lg.get("i_ptm", lg.get("ptm", 0.0))), 3),                 plddt=round(float(lg.get("plddt", 0.0)), 3),                 hydropathy=round(hydropathy(seq), 3))        out = "%s/%s.pdb" % (outdir, name)        model.save_pdb(out)        # Geometry is measured in its own guarded step: a failure here must not discard a        # design whose confidence numbers are already in hand.        try:            r.update(measure(out, L))        except Exception as e:            r["measure_error"] = "%s %s" % (type(e).__name__, str(e)[:160])        with open(rec_path, "a") as fh:            fh.write(json.dumps(r) + "\n")        made += 1        extra = ""        if MODE == "thread" and "deepest" in r:            extra = " deep=%.1f nterm=%.1f %s %dTM" % (                r["deepest"], r["nterm_depth"],                "THREADED" if r["threaded"] else "backwards", r["tm_contacts"])        log("%s len=%d i_ptm=%.3f pLDDT=%.3f kd=%+.2f hlx=%.2f on_site=%d%s"            % (name, L, r["true_iptm"], r["plddt"], r["hydropathy"],               r.get("helicity", 0.0), len(r.get("on_site", [])), extra))    except Exception as e:        log("%s FAILED %s %s" % (name, type(e).__name__, str(e)[:200]))log("done: %d designs saved from %d attempts" % (made, i))if made == 0:    raise SystemExit("worker %s produced nothing - read the failures above" % MODE)'''os.makedirs("/kaggle/working/glp1r_ecd", exist_ok=True)os.makedirs("/kaggle/working/glp1r_thread", exist_ok=True)with open("/kaggle/working/glp1r_worker.py", "w") as fh:    fh.write(textwrap.dedent(WORKER))procs = []for gpu, mode in (("0", "ecd"), ("1", "thread")):    p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/glp1r_worker.py",                          gpu, mode, str(TIME_BUDGET_MIN),                          str(SOFT), str(TEMP), str(HARD)],                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                         text=True, bufsize=1)    procs.append(p)    print("launched %s job on GPU %s" % (mode, gpu), flush=True)import threadingdef pump(p):    for line in p.stdout:        print(line.rstrip(), flush=True)threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs][t.start() for t in threads]for p in procs:    p.wait()[t.join(timeout=10) for t in threads]import glob, jsonrecs = []for f in sorted(glob.glob("/kaggle/working/glp1r_*/designs.jsonl")):    for line in open(f):        line = line.strip()        if line:            recs.append(json.loads(line))json.dump(recs, open("/kaggle/working/glp1r_designs.json", "w"), indent=1)if not recs:    raise SystemExit("no designs at all from either job - this run produced nothing")for mode in ("ecd", "thread"):    sub = sorted([r for r in recs if r["mode"] == mode], key=lambda r: -r["true_iptm"])    print("\n==== %s: %d designs ====" % (mode.upper(), len(sub)))    hdr = "%-18s %4s %7s %7s %7s %6s %7s" % ("name", "len", "i_ptm", "pLDDT",                                             "kd", "hlx", "on_site")    print(hdr + ("  depth  nterm  threaded" if mode == "thread" else ""))    for r in sub[:25]:        line = "%-18s %4d %7.3f %7.3f %+7.2f %6.2f %7d" % (            r["name"], r["length"], r["true_iptm"], r["plddt"], r["hydropathy"],            r.get("helicity", 0.0), len(r.get("on_site", [])))        if mode == "thread":            line += "  %5.1f  %5.1f  %s" % (r.get("deepest", 0.0),                                            r.get("nterm_depth", 0.0),                                            r.get("threaded", False))        print(line)    if sub:        good = [r for r in sub if r["true_iptm"] >= 0.50]        onsite = [r for r in sub if len(r.get("on_site", [])) >= 5]        cool = [r for r in sub if r["hydropathy"] <= 0.0]        print("%d at i_ptm>=0.50, %d contact 5+ intended residues, %d not greasy (kd<=0)"              % (len(good), len(onsite), len(cool)))        if mode == "thread":            thr = [r for r in sub if r.get("threaded")]            print("%d of %d are threaded the right way round (N-terminus deepest)"                  % (len(thr), len(sub)))print("\n===== COPY FROM HERE =====")for r in recs:    print(json.dumps(r, separators=(",", ":")))print("===== COPY TO HERE =====")